# Trade Simulation - Realistic Execution Windows

Simulate institutional equity order execution with realistic timing, multiple fills, venue routing, and participation rates.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

## Load Market Data

In [ ]:
# Load processed market data from previous step
market_data = pd.read_csv('market_data_processed.csv')

market_data['Datetime'] = pd.to_datetime(market_data['Datetime'])
market_data['Date'] = pd.to_datetime(market_data['Date'])

print(f"Loaded {len(market_data):,} market data bars")
print(f"Tickers: {market_data['Ticker'].nunique()}")
print(f"Date range: {market_data['Datetime'].min().date()} to {market_data['Datetime'].max().date()}")

## Generate Parent Orders

Parent orders represent the initial client orders before execution.

In [ ]:
def generate_parent_orders(market_data, num_orders=200):
    """Generate realistic parent orders with varied sizes and strategies."""
    
    # Sample up to 3:30 PM to allow some Market Close arrivals
    eligible_bars = market_data[market_data['Datetime'].dt.hour <= 15].copy()
    sampled_bars = eligible_bars.sample(n=num_orders, replace=True).reset_index(drop=True)
    
    orders = pd.DataFrame()
    orders['OrderID'] = range(1, num_orders + 1)
    orders['Ticker'] = sampled_bars['Ticker'].values
    orders['Date'] = sampled_bars['Date'].values
    orders['ArrivalTime'] = sampled_bars['Datetime'].values
    orders['ArrivalPrice'] = sampled_bars['Close'].values
    orders['ArrivalSession'] = sampled_bars['Session'].values
    orders['AvgDailyVolume'] = sampled_bars['AvgDailyVolume'].values
    orders['AvgVolatility'] = sampled_bars['AvgVolatility'].values
    
    # Order side
    orders['Side'] = np.random.choice(['Buy', 'Sell'], size=num_orders)
    
    # Order size categories — target ~$100-200M total notional across 200 orders
    size_types = np.random.choice(['small', 'medium', 'large', 'very_large'],
                                  size=num_orders,
                                  p=[0.40, 0.35, 0.20, 0.05])
    
    def get_order_quantity(size_type):
        if size_type == 'small':
            return np.random.randint(100, 500)
        elif size_type == 'medium':
            return np.random.randint(500, 3000)
        elif size_type == 'large':
            return np.random.randint(3000, 10000)
        else:
            return np.random.randint(10000, 30000)
    
    orders['Quantity'] = [get_order_quantity(s) for s in size_types]
    orders['SizeType'] = size_types
    orders['PctOfADV'] = (orders['Quantity'] / orders['AvgDailyVolume']) * 100
    
    # Assign execution strategy based on order size category
    # Larger orders use more patient strategies; small orders can cross immediately
    strategy_probs = {
        'small':      ['Immediate', 'VWAP', 'VWAP-Dark', 'Careful'],
        'medium':     ['Immediate', 'VWAP', 'VWAP-Dark', 'Careful'],
        'large':      ['Immediate', 'VWAP', 'VWAP-Dark', 'Careful'],
        'very_large': ['Immediate', 'VWAP', 'VWAP-Dark', 'Careful'],
    }
    strategy_weights = {
        'small':      [0.40, 0.40, 0.15, 0.05],
        'medium':     [0.10, 0.40, 0.35, 0.15],
        'large':      [0.00, 0.20, 0.45, 0.35],
        'very_large': [0.00, 0.10, 0.40, 0.50],
    }
    
    def assign_strategy(size_type):
        return np.random.choice(strategy_probs[size_type], p=strategy_weights[size_type])
    
    orders['Strategy'] = [assign_strategy(s) for s in size_types]
    
    # Participation rate
    def get_participation_rate(strategy):
        if strategy == 'Immediate':
            return np.random.uniform(20, 40)
        elif strategy == 'VWAP':
            return np.random.uniform(10, 20)
        elif strategy == 'VWAP-Dark':
            return np.random.uniform(8, 15)
        else:
            return np.random.uniform(5, 10)
    
    orders['ParticipationRate'] = orders['Strategy'].apply(get_participation_rate)
    
    return orders


print("Generating parent orders...")
parent_orders = generate_parent_orders(market_data, num_orders=200)

print(f"Generated {len(parent_orders)} parent orders")
print("\nStrategy distribution:")
print(parent_orders['Strategy'].value_counts())

## Simulate Execution Over Time

Large orders are broken into multiple child fills and executed across realistic time windows.

In [ ]:
def calculate_fill_slippage(order, fill_qty, interval_volume, num_fills, fill_index):
    """
    Calculate slippage for a single fill based on order characteristics.
    
    Factors: fill size vs volume, volatility, urgency, strategy type.
    """
    
    # Base slippage
    base_slippage = np.random.uniform(0.5, 2.0)
    
    # Fill size impact
    fill_pct_of_volume = (fill_qty / interval_volume) * 100 if interval_volume > 0 else 0.01
    
    if fill_pct_of_volume < 0.1:
        volume_impact = np.random.uniform(0.2, 1.0)
    elif fill_pct_of_volume < 0.5:
        volume_impact = np.random.uniform(1.0, 3.0)
    elif fill_pct_of_volume < 1.0:
        volume_impact = np.random.uniform(3.0, 6.0)
    else:
        volume_impact = np.random.uniform(6.0, 12.0)
    
    # Volatility impact
    volatility = order['AvgVolatility']
    if volatility > 0.5:
        volatility_impact = np.random.uniform(1.5, 3.0)
    elif volatility > 0.3:
        volatility_impact = np.random.uniform(0.8, 1.5)
    else:
        volatility_impact = np.random.uniform(0.3, 0.8)
    
    # Urgency impact (early fills more aggressive = higher cost)
    urgency_factor = 1.0 - (fill_index / num_fills) * 0.3
    
    # Strategy impact — patient algos have lower per-fill market impact
    if order['Strategy'] == 'Immediate':
        strategy_multiplier = 1.3
    elif order['Strategy'] == 'VWAP':
        strategy_multiplier = 0.6
    elif order['Strategy'] == 'VWAP-Dark':
        strategy_multiplier = 0.5
    else:  # Careful
        strategy_multiplier = 0.3
    
    total_slippage = (base_slippage + volume_impact + volatility_impact) * urgency_factor * strategy_multiplier
    
    return total_slippage


def simulate_execution_window(order, market_data):
    """Simulate execution of a single parent order over time."""
    
    # Determine execution duration based on strategy
    if order['Strategy'] == 'Immediate':
        duration_minutes = np.random.randint(5, 20)
    elif order['Strategy'] == 'VWAP':
        duration_minutes = np.random.randint(60, 180)
    elif order['Strategy'] == 'VWAP-Dark':
        duration_minutes = np.random.randint(120, 240)
    else:  # Careful
        duration_minutes = np.random.randint(180, 360)
    
    exec_start = order['ArrivalTime']
    exec_end = exec_start + timedelta(minutes=duration_minutes)
    
    # Cap at market close
    market_close = exec_start.replace(hour=16, minute=0, second=0, microsecond=0)
    if exec_end > market_close:
        exec_end = market_close
        duration_minutes = int((exec_end - exec_start).total_seconds() / 60)
    
    # Get market data during execution window
    exec_window_data = market_data[
        (market_data['Ticker'] == order['Ticker']) &
        (market_data['Date'] == order['Date']) &
        (market_data['Datetime'] >= exec_start) &
        (market_data['Datetime'] <= exec_end)
    ].copy()
    
    if len(exec_window_data) == 0:
        exec_window_data = market_data[
            (market_data['Ticker'] == order['Ticker']) &
            (market_data['Datetime'] == exec_start)
        ].copy()
    
    # Calculate interval VWAP during execution window
    if len(exec_window_data) > 0:
        exec_window_data['TypicalPrice'] = (exec_window_data['High'] + exec_window_data['Low'] + exec_window_data['Close']) / 3
        exec_window_data['PV'] = exec_window_data['TypicalPrice'] * exec_window_data['Volume']
        interval_vwap = exec_window_data['PV'].sum() / exec_window_data['Volume'].sum()
        interval_volume = exec_window_data['Volume'].sum()
    else:
        interval_vwap = order['ArrivalPrice']
        interval_volume = order['AvgDailyVolume'] / 78
    
    # Number of fills driven by execution duration
    # Immediate: 1-2 fills, just cross the spread
    # Algo orders: roughly one fill per 10 minutes of execution
    if order['Strategy'] == 'Immediate':
        num_fills = min(np.random.randint(1, 3), len(exec_window_data))
    else:
        num_fills = min(max(2, duration_minutes // 10), len(exec_window_data))
    
    num_fills = max(num_fills, 1)
    
    # Pre-select fill bars: evenly spaced across the execution window (like a real slicing algo)
    # This ensures early fills are near arrival and later fills follow time order
    exec_window_data = exec_window_data.sort_values('Datetime').reset_index(drop=True)
    bar_indices = np.linspace(0, len(exec_window_data) - 1, num_fills, dtype=int)
    
    # Generate child fills
    fills = []
    remaining_qty = order['Quantity']
    
    for i in range(num_fills):
        if i == num_fills - 1:
            fill_qty = remaining_qty
        else:
            target_fill = order['Quantity'] / num_fills
            fill_qty = int(np.random.uniform(0.7, 1.3) * target_fill)
            fill_qty = min(fill_qty, remaining_qty)
        
        # Pick the fill bar in time order
        if len(exec_window_data) > 0:
            fill_bar = exec_window_data.iloc[bar_indices[i]]
            fill_time = fill_bar['Datetime']
            fill_price_base = fill_bar['Close']
        else:
            fill_time = exec_start
            fill_price_base = order['ArrivalPrice']
        
        # Calculate slippage based on order characteristics
        fill_slippage_bps = calculate_fill_slippage(
            order, fill_qty, interval_volume, num_fills, i
        )
        
        if order['Side'] == 'Buy':
            fill_price = fill_price_base * (1 + fill_slippage_bps / 10000)
        else:
            fill_price = fill_price_base * (1 - fill_slippage_bps / 10000)
        
        # Venue selection
        venue = 'Dark' if order['Strategy'] in ['VWAP-Dark', 'Careful'] and np.random.rand() < 0.4 else 'Lit'
        
        fills.append({
            'FillNum': i + 1,
            'FillTime': fill_time,
            'FillQty': fill_qty,
            'FillPrice': fill_price,
            'FillSlippageBps': fill_slippage_bps,
            'Venue': venue
        })
        
        remaining_qty -= fill_qty
    
    # Calculate weighted average execution price
    total_qty = sum(f['FillQty'] for f in fills)
    weighted_exec_price = sum(f['FillPrice'] * f['FillQty'] for f in fills) / total_qty
    
    lit_pct = sum(f['FillQty'] for f in fills if f['Venue'] == 'Lit') / total_qty * 100 if total_qty > 0 else 100
    
    return {
        'ExecStartTime': exec_start,
        'ExecEndTime': exec_end,
        'ExecDurationMins': duration_minutes,
        'NumFills': len(fills),
        'AvgExecPrice': weighted_exec_price,
        'IntervalVWAP': interval_vwap,
        'LitPct': lit_pct,
        'DarkPct': 100 - lit_pct,
        'Fills': fills
    }

In [ ]:
# Simulate execution for all parent orders
print("Simulating realistic execution windows...")
print("This may take a minute...\n")

execution_results = []

for idx, order in parent_orders.iterrows():
    if (idx + 1) % 50 == 0:
        print(f"  Processing order {idx + 1}/{len(parent_orders)}...")
    
    exec_result = simulate_execution_window(order, market_data)
    execution_results.append(exec_result)

# Add results back to parent_orders
parent_orders['ExecStartTime'] = [r['ExecStartTime'] for r in execution_results]
parent_orders['ExecEndTime'] = [r['ExecEndTime'] for r in execution_results]
parent_orders['ExecDurationMins'] = [r['ExecDurationMins'] for r in execution_results]
parent_orders['NumFills'] = [r['NumFills'] for r in execution_results]
parent_orders['AvgExecPrice'] = [r['AvgExecPrice'] for r in execution_results]
parent_orders['IntervalVWAP'] = [r['IntervalVWAP'] for r in execution_results]
parent_orders['LitPct'] = [r['LitPct'] for r in execution_results]
parent_orders['DarkPct'] = [r['DarkPct'] for r in execution_results]

print(f"\nExecution simulation complete for {len(parent_orders)} orders")

## Calculate Performance Metrics

In [ ]:
# Implementation Shortfall (vs Arrival Price)
parent_orders['IS_Bps'] = ((parent_orders['AvgExecPrice'] - parent_orders['ArrivalPrice']) / parent_orders['ArrivalPrice']) * 10000

# Adjust sign based on side (positive = cost)
parent_orders.loc[parent_orders['Side'] == 'Sell', 'IS_Bps'] = -parent_orders.loc[parent_orders['Side'] == 'Sell', 'IS_Bps']

# Clamp IS to realistic range — extreme values are artifacts of bar sampling
parent_orders['IS_Bps'] = parent_orders['IS_Bps'].clip(-25, 25)

# Performance vs Interval VWAP
parent_orders['VsIntervalVWAP_Bps'] = ((parent_orders['AvgExecPrice'] - parent_orders['IntervalVWAP']) / parent_orders['IntervalVWAP']) * 10000
parent_orders.loc[parent_orders['Side'] == 'Sell', 'VsIntervalVWAP_Bps'] = -parent_orders.loc[parent_orders['Side'] == 'Sell', 'VsIntervalVWAP_Bps']

# Dollar cost and notional
parent_orders['TotalCost_Dollars'] = abs(parent_orders['AvgExecPrice'] - parent_orders['ArrivalPrice']) * parent_orders['Quantity']
parent_orders['NotionalValue'] = parent_orders['Quantity'] * parent_orders['ArrivalPrice']

print("Performance metrics calculated")

## Summary Statistics

In [7]:
print("="*80)
print("EXECUTION SIMULATION SUMMARY")
print("="*80)

print(f"\nTotal Orders: {len(parent_orders)}")
print(f"Total Notional: ${parent_orders['NotionalValue'].sum():,.0f}\n")

print("--- Implementation Shortfall (vs Arrival) ---")
print(f"Average IS: {parent_orders['IS_Bps'].mean():.2f} bps")
print(f"Median IS: {parent_orders['IS_Bps'].median():.2f} bps")
print(f"Total Cost: ${parent_orders['TotalCost_Dollars'].sum():,.2f}\n")

print("--- Performance vs Interval VWAP ---")
print(f"Average: {parent_orders['VsIntervalVWAP_Bps'].mean():.2f} bps")
print(f"Orders beating Interval VWAP: {(parent_orders['VsIntervalVWAP_Bps'] < 0).sum()} ({(parent_orders['VsIntervalVWAP_Bps'] < 0).sum()/len(parent_orders)*100:.1f}%)\n")

print("--- Execution Characteristics ---")
print(f"Avg execution duration: {parent_orders['ExecDurationMins'].mean():.0f} minutes")
print(f"Avg number of fills: {parent_orders['NumFills'].mean():.1f}")
print(f"Avg lit execution: {parent_orders['LitPct'].mean():.1f}%")
print(f"Avg dark execution: {parent_orders['DarkPct'].mean():.1f}%\n")

print("--- By Strategy ---")
strategy_summary = parent_orders.groupby('Strategy').agg({
    'OrderID': 'count',
    'IS_Bps': 'mean',
    'ExecDurationMins': 'mean',
    'NumFills': 'mean',
    'DarkPct': 'mean'
}).round(2)
strategy_summary.columns = ['Orders', 'Avg_IS_Bps', 'Avg_Duration_Mins', 'Avg_Fills', 'Avg_Dark_Pct']
print(strategy_summary)

print("\n" + "="*80)

EXECUTION SIMULATION SUMMARY

Total Orders: 200
Total Notional: $32,495,623,656

--- Implementation Shortfall (vs Arrival) ---
Average IS: 6.19 bps
Median IS: 5.74 bps
Total Cost: $97,236,431.72

--- Performance vs Interval VWAP ---
Average: 4.11 bps
Orders beating Interval VWAP: 51 (25.5%)

--- Execution Characteristics ---
Avg execution duration: 95 minutes
Avg number of fills: 7.0
Avg lit execution: 87.7%
Avg dark execution: 12.3%

--- By Strategy ---
           Orders  Avg_IS_Bps  Avg_Duration_Mins  Avg_Fills  Avg_Dark_Pct
Strategy                                                                 
Careful        33        1.24             227.15      14.76         38.59
Immediate      90        5.51              12.11       2.69          0.00
VWAP           48       14.91             117.27       7.48          0.00
VWAP-Dark      29       -0.51             163.69      10.41         40.93



## Save Simulated Data

In [ ]:
# Save results
output_file = 'simulated_trades.csv'
parent_orders.to_csv(output_file, index=False)

print(f"Simulated trades saved to: {output_file}")
print(f"Total records: {len(parent_orders):,}")